# 🧪 DeepEval & LangChain: Production RAG System Evaluation with Google Gemini

Welcome to the comprehensive masterclass on **RAG System Evaluation** using **DeepEval** (by *Confident AI*) and **LangChain** powered by **Google Gemini**.

---

### 🎯 Learning Objectives:
1. **Understand the DeepEval Framework**: Learn how DeepEval functions as a unit-testing harness for LLMs (Pytest for AI applications).
2. **Master the RAG Triad & Extended Metrics**: Understand the 5 core RAG evaluation dimensions:
   - **Answer Relevancy**: Does the output address the query directly?
   - **Faithfulness**: Is the response strictly grounded in retrieved context without hallucination?
   - **Contextual Relevancy**: Are retrieved document chunks relevant to the query?
   - **Contextual Precision**: Are the most relevant chunks ranked at the top?
   - **Contextual Recall**: Does the retrieved context satisfy the ground-truth expected answer?
3. **Build an End-to-End Grounded RAG App**: Construct an in-memory retrieval chain using `LangChain`, `GoogleGenerativeAIEmbeddings`, `InMemoryVectorStore`, and `ChatGoogleGenerativeAI` (`gemini-3.1-flash-lite`).
4. **Configure a Native Gemini Judge**: Use DeepEval's `GeminiModel` powered by `gemini-3-flash-preview` to perform rigorous LLM-as-a-Judge evaluations.
5. **Analyze Diagnostic Reports**: Parse quantitative scores ($0.0 - 1.0$), inspect chain-of-thought judge rationales, and export results to CSV for regression testing.

---

### 🔗 Official Documentation & References:
* **GitHub Repository**: [https://github.com/confident-ai/deepeval](https://github.com/confident-ai/deepeval)
* **DeepEval Documentation**: [https://deepeval.com/](https://deepeval.com/)
* **Confident AI Platform**: [https://www.confident-ai.com/](https://www.confident-ai.com/)

## 1. DeepEval Metric Ecosystem Taxonomy

DeepEval organizes evaluation metrics and testing utilities across specialized functional categories:

### 1.1 Core Evaluation & Testing Primitives
Essential test-case wrappers, datasets, and assertion runners:
```python
from deepeval import evaluate, assert_test

from deepeval.dataset import (
    EvaluationDataset,
    Golden,
)

from deepeval.test_case import (
    LLMTestCase,
    ConversationalTestCase,
    Turn,
    ToolCall,
    SingleTurnParams,
    ArenaTestCase,
    Contestant,
)
```

---

### 1.2 RAG Evaluation Metrics (The Core 5)
Comprehensive metrics assessing both the retrieval engine and the generation model:
* **AnswerRelevancyMetric**: Measures if the generated answer directly addresses the user query.
* **FaithfulnessMetric**: Verifies that generated claims are fully supported by retrieved context (hallucination detector).
* **ContextualRelevancyMetric**: Evaluates the signal-to-noise ratio in retrieved context chunks.
* **ContextualPrecisionMetric**: Evaluates if the most relevant chunks are ranked at the top of the retrieval context.
* **ContextualRecallMetric**: Evaluates if the context retrieved contains all information necessary to match the ground truth.

```python
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
```

---

### 1.3 Agent & Tool Calling Evaluation Metrics
Specialized metrics for multi-step autonomous agents, tool orchestration, and multi-turn dialogue:
* **Planning & Reasoning**: `PlanQualityMetric`, `PlanAdherenceMetric`
* **Tool & Argument Accuracy**: `ToolCorrectnessMetric`, `ArgumentCorrectnessMetric`, `ToolUseMetric`
* **Execution Success**: `TaskCompletionMetric`, `StepEfficiencyMetric`, `GoalAccuracyMetric`
* **Conversational Quality**: `ConversationCompletenessMetric`, `ConversationRelevancyMetric`, `RoleAdherenceMetric`, `KnowledgeRetentionMetric`

```python
from deepeval.metrics import (
    # Reasoning / Planning
    PlanQualityMetric,
    PlanAdherenceMetric,

    # Tool / Action
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,

    # Overall Execution
    TaskCompletionMetric,
    StepEfficiencyMetric,
    ToolUseMetric,
    GoalAccuracyMetric,

    # Conversational Dynamics
    ConversationCompletenessMetric,
    ConversationRelevancyMetric,
    RoleAdherenceMetric,
    KnowledgeRetentionMetric,
)
```

---

### 1.4 Safety & Guardrail Evaluation Metrics
Metrics dedicated to trust, safety, and compliance verification:
* **BiasMetric**: Detects unfair stereotypes, demographic biases, and discrimination.
* **ToxicityMetric**: Flags hate speech, aggression, insults, and harassment.
* **NonAdviceMetric**: Verifies that the model refuses unauthorized legal, medical, or financial advice.
* **MisuseMetric**: Detects attempts to elicit illegal or harmful procedural knowledge.
* **PIILeakageMetric**: Flags unauthorized disclosure of personally identifiable information.
* **RoleViolationMetric**: Evaluates breaches in operational persona boundaries.

```python
from deepeval.metrics import (
    BiasMetric,
    ToxicityMetric,
    NonAdviceMetric,
    MisuseMetric,
    PIILeakageMetric,
    RoleViolationMetric,
)
```

---

### 1.5 General LLM & Structured Output Quality Metrics
Metrics evaluating fundamental LLM generation characteristics:
* **HallucinationMetric**: Measures factual fidelity against known baseline assertions.
* **PromptAlignmentMetric**: Validates strict instruction adherence.
* **JsonCorrectnessMetric**: Verifies valid JSON syntax and schema compliance.
* **SummarizationMetric**: Evaluates concise preservation of original text content.
* **ExactMatchMetric**: Evaluates deterministic string equality.

```python
from deepeval.metrics import (
    HallucinationMetric,
    PromptAlignmentMetric,
    JsonCorrectnessMetric,
    SummarizationMetric,
    ExactMatchMetric,
)
```

---

### 1.6 Custom LLM-as-a-Judge & Arena Evaluation
Frameworks for constructing custom criteria and head-to-head model comparisons:
* **GEval**: DeepEval's proprietary criteria-based evaluation algorithm using custom scoring rubrics.
* **DAGMetric**: Graph-based evaluation workflows for complex, conditional multi-step criteria.
* **ArenaGEval**: Side-by-side comparative evaluation between competing models.

```python
from deepeval.metrics import (
    GEval,
    DAGMetric,
    ArenaGEval,
)
```

---

### 1.7 Tracing & Observability
DeepEval's built-in tracing utilities for production application monitoring and span tracking:
```python
from deepeval.tracing import (
    observe,
    update_current_span,
)
```

## 2. End-to-End System Architecture

The evaluation architecture is composed of two decoupled pipelines:
1. **LangChain RAG Pipeline**: Chunks are embedded via `GoogleGenerativeAIEmbeddings` into `InMemoryVectorStore`. A query retrieves top-$k$ relevant chunks, constructs a grounded prompt, and passes it to `ChatGoogleGenerativeAI` (`gemini-3.1-flash-lite`) to produce the answer.
2. **DeepEval Evaluation Pipeline**: The generated answer, query, context, and ground truth are packaged into an `LLMTestCase`. An independent Evaluator Judge (`gemini-3-flash-preview` via `GeminiModel`) evaluates the test case across the 5 core RAG metrics, returning numerical scores ($0.0 - 1.0$), pass/fail statuses, and chain-of-thought reasons.

```text
                                      RAG
                                       │
              ┌────────────────────────┼────────────────────────┐
              │                        │                        │
GoogleGenerativeAIEmbeddings      Vector Store              Retriever
                                                                │
                                                                ▼
                                                         Retrieved Docs
                                                                │
                                                                ▼
                                                       ChatPromptTemplate
                                                                │
                                                                ▼
                                                     ChatGoogleGenerativeAI
                                                                │
                                                                ▼
                                                         StrOutputParser
                                                                │
                                                                ▼
                                                            Response
                                                                │
                                                                │
                                DEEPEVAL EVALUATION             │
                                       │                        │
                                       ▼                        ▼
                                 Golden Dataset              Response
                                       │                        │
                                       └───────────┬────────────┘
                                                   ▼
                                              LLMTestCase
                                                   │
                                                   ▼
                                       DeepEval GeminiModel Judge
                                                   │
                ┌──────────────────────────────────┼──────────────────────────────────┐
                │                  │               │                  │               │
                ▼                  ▼               ▼                  ▼               ▼
         AnswerRelevancy      Faithfulness   ContextRelevancy   ContextPrecision ContextRecall
                │                  │               │                  │               │
                └──────────────────┴───────────────┬──────────────────┴───────────────┘
                                                   ▼
                                             Score + Reason
                                                   │
                                                   ▼
                                           Experiment Results
                                                   │
                                                   ▼
                                                  CSV
```

---
## 3. Environment Setup & API Configuration

We load environment variables from `.env` and verify API key availability.
* We configure **`GOOGLE_API_KEY`** (with fallback to `GEMINI_API_KEY`) and set both in `os.environ` so `langchain_google_genai`, `google.genai`, and DeepEval's `GeminiModel` discover it automatically.

In [2]:
import os
import asyncio
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [3]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY or GEMINI_API_KEY is missing. Please add it to the .env file."
    )

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY

print("API key loaded successfully.")

API key loaded successfully.


---
## 4. Synthetic Knowledge Base & In-Memory Vector Store

We define a curated synthetic knowledge base about **Ragas** and **RAG systems**.
Each document is wrapped in a LangChain `Document` object:
* **`page_content`**: The raw text that will be embedded and retrieved.
* **`metadata` (`id`)**: Unique document identifier for retrieval tracking and provenance.

In [4]:
documents = [
    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={"id": "ragas_intro"},
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications.

Experiments can track evaluation results,
compare different application versions,
and store experiment outputs.
""".strip(),
        metadata={"id": "ragas_experiments"},
    ),

    Document(
        page_content="""
Ragas metrics measure different properties
of AI systems.

Examples include factual correctness,
faithfulness, answer relevancy,
retrieval quality, and agent behavior.
""".strip(),
        metadata={"id": "ragas_metrics"},
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge source and provides that
information to an LLM before generating
the final answer.
""".strip(),
        metadata={"id": "rag_intro"},
    ),
]

### 4.1 Vector Store Indexing & Retrieval
We embed documents using **`GoogleGenerativeAIEmbeddings`** (`gemini-embedding-001`, 3072 dimensions) and index them in an **`InMemoryVectorStore`**.
We configure the retriever to fetch top-$k=2$ chunks per query.

In [5]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [6]:
vector_store = InMemoryVectorStore(
    embedding=embedding_model,
)

vector_store.add_documents(documents)

['7b927e57-dc16-459e-b636-07ccdd16ebf6',
 'e1642fe9-91c3-412a-965f-f2b62d9f800a',
 '6241bbe1-a12b-4d28-8534-75478cb9c51d',
 'a5d93d3a-c362-4467-b0f5-80e1d4c7679e']

In [7]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
)

---
## 5. Grounded Prompt Engineering & LCEL RAG Chain

We configure the RAG generation pipeline:
* **Strict Grounding**: Instructs the model to answer *only* using provided context and explicitly state when information is insufficient.
* **Generator LLM**: `ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)` for deterministic generation.
* **LCEL Assembly**: `rag_prompt | llm | StrOutputParser()`.

In [8]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question using the provided context.

If the context does not contain enough information,
say that you do not have enough information.

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [10]:
rag_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

---
## 6. End-to-End RAG Query Application

The asynchronous function `rag_query(question)` encapsulates the runtime RAG pipeline:
1. **Retrieval**: `await retriever.ainvoke(question)` fetches top context chunks and IDs.
2. **Context Formatting**: Chunks are joined into a clean context string.
3. **Generation**: `await rag_chain.ainvoke()` produces the final grounded answer.
4. **Structured Payload**: Returns `answer`, `retrieved_contexts`, and `retrieved_document_ids`.

In [11]:
async def rag_query(question):

    retrieved_documents = await retriever.ainvoke(question)

    retrieved_contexts = [
        document.page_content
        for document in retrieved_documents
    ]

    retrieved_document_ids = [
        document.metadata["id"]
        for document in retrieved_documents
    ]

    context = "\n\n".join(retrieved_contexts)

    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,
    }

### 6.1 Interactive Sanity Check
A test execution to verify that retrieval and generation work end-to-end before running the evaluation benchmark.

In [12]:
question = "What is RAG and how does it work?"

In [13]:
await rag_query(question)

{'answer': 'RAG stands for Retrieval-Augmented Generation. It works by retrieving relevant information from a knowledge source and providing that information to an LLM before the final answer is generated.',
 'retrieved_contexts': ['RAG stands for Retrieval-Augmented Generation.\n\nA RAG system retrieves relevant information\nfrom a knowledge source and provides that\ninformation to an LLM before generating\nthe final answer.',
  'Ragas metrics measure different properties\nof AI systems.\n\nExamples include factual correctness,\nfaithfulness, answer relevancy,\nretrieval quality, and agent behavior.'],
 'retrieved_document_ids': ['rag_intro', 'ragas_metrics']}

---
## 7. The DeepEval RAG Evaluation Paradigm (The RAG Triad)

Evaluating Retrieval-Augmented Generation requires assessing two distinct subsystems: the **Retriever** and the **Generator**.
DeepEval structures this assessment around the **RAG Triad** and extended contextual metrics:

```text
               ┌────────────────────────┐
               │         Query          │
               └───────────┬────────────┘
                           │
             Contextual    │    Answer
             Relevancy     │    Relevancy
                           │
          ┌────────────────┴────────────────┐
          ▼                                 ▼
┌──────────────────┐  Faithfulness  ┌──────────────────┐
│ Retrieved Context│◄───────────────┤ Generated Answer │
└──────────────────┘                └──────────────────┘
```

---

### The 5 Core DeepEval RAG Metrics:

| Metric | Target Subsystem | Evaluation Question | Required `LLMTestCase` Fields |
| :--- | :--- | :--- | :--- |
| **1. Answer Relevancy** | **Generator** | Does the actual output answer the user query directly and concisely without irrelevant chatter? | `input`, `actual_output` |
| **2. Faithfulness** | **Generator** | Is every claim in the output factually supported by the retrieved context (absence of hallucinations)? | `actual_output`, `retrieval_context` |
| **3. Contextual Relevancy** | **Retriever** | Does the retrieved context contain only relevant information, or is it contaminated with noise? | `input`, `retrieval_context` |
| **4. Contextual Precision** | **Retriever** | Are the most relevant context chunks ranked near the top of the retrieval list? | `input`, `actual_output`, `expected_output`, `retrieval_context` |
| **5. Contextual Recall** | **Retriever** | Does the retrieved context cover all the critical information present in the expected ground truth? | `expected_output`, `retrieval_context` |

---

### 💡 DeepEval Evaluation Settings:
* **Threshold (`threshold=0.7`)**: DeepEval scores range continuously from $0.0$ to $1.0$. A score $\ge 0.70$ is considered a `PASS`.
* **Reasoning (`include_reason=True`)**: Instructs the Judge LLM to provide a chain-of-thought natural language explanation justifying the score, invaluable for debugging retrieval noise or generator hallucination.

---
## 8. Golden Evaluation Dataset (Ground Truth)

A **Golden Dataset** provides human-curated or verified baseline pairs (`question`, `expected_answer`).
This ground truth serves as the reference standard against which `ContextualPrecisionMetric`, `ContextualRecallMetric`, and `AnswerRelevancyMetric` evaluate runtime performance.

In [14]:
# ============================================================
# 11. LOAD EVALUATION DATASET
# ============================================================

def load_dataset():

    return [
        {
            "question": "What is Ragas?",
            "expected_answer": """
Ragas is an evaluation framework
for LLM applications.
""".strip(),
        },

        {
            "question": "How do Ragas experiments work?",
            "expected_answer": """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),
        },

        {
            "question": "What do Ragas metrics do?",
            "expected_answer": """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),
        },

        {
            "question": "What is RAG?",
            "expected_answer": """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),
        },
    ]

In [15]:
load_dataset()

[{'question': 'What is Ragas?',
  'expected_answer': 'Ragas is an evaluation framework\nfor LLM applications.'},
 {'question': 'How do Ragas experiments work?',
  'expected_answer': 'Ragas experiments provide systematic evaluation,\ntrack results, compare different application versions,\nand store experiment outputs.'},
 {'question': 'What do Ragas metrics do?',
  'expected_answer': 'Ragas metrics evaluate properties and quality\nof AI or LLM application outputs.'},
 {'question': 'What is RAG?',
  'expected_answer': 'RAG stands for Retrieval-Augmented Generation.\n\nIt retrieves relevant information from a knowledge\nsource and provides that context to an LLM\nbefore answer generation.'}]

---
## 9. DeepEval Gemini Evaluator & Metric Configuration

We configure the DeepEval evaluation harness:
* **Judge LLM (`GeminiModel`)**: DeepEval's native integration with Google GenAI using `gemini-3-flash-preview` for high-reasoning evaluation.
* **Metric Suite**: We instantiate the 5 RAG metrics with `threshold=0.7`, `model=judge_model`, and `include_reason=True`.

In [16]:
from deepeval.models import GeminiModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [17]:
judge_model = GeminiModel(
    model="gemini-3-flash-preview",
    api_key=os.environ["GOOGLE_API_KEY"],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [18]:
def create_metrics():

    answer_relevancy = AnswerRelevancyMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    faithfulness = FaithfulnessMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_relevancy = ContextualRelevancyMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_precision = ContextualPrecisionMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_recall = ContextualRecallMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    return {
        "answer_relevancy": answer_relevancy,
        "faithfulness": faithfulness,
        "contextual_relevancy": contextual_relevancy,
        "contextual_precision": contextual_precision,
        "contextual_recall": contextual_recall,
    }

---
## 10. Single Test-Case Evaluation & LLMTestCase Contract

DeepEval standardizes evaluation inputs through the **`LLMTestCase`** data contract:
* **`input`**: The user query string.
* **`actual_output`**: The live answer generated by `rag_chain`.
* **`expected_output`**: The ground-truth answer from the golden dataset.
* **`retrieval_context`**: List of raw text chunks fetched by the retriever.

The function `evaluate_test_case(row)` runs the RAG pipeline, constructs the `LLMTestCase`, and asynchronously evaluates each of the 5 metrics via `await metric.a_measure(test_case)`.

In [19]:
async def evaluate_test_case(row):

    question = row["question"]
    expected_answer = row["expected_answer"]

    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    rag_result = await rag_query(question)

    response = rag_result["answer"]
    retrieved_contexts = rag_result["retrieved_contexts"]
    retrieved_document_ids = rag_result["retrieved_document_ids"]

    # ========================================================
    # CREATE DEEPEVAL TEST CASE
    # ========================================================

    test_case = LLMTestCase(
        input=question,
        actual_output=response,
        expected_output=expected_answer,
        retrieval_context=retrieved_contexts,
    )

    # ========================================================
    # CREATE METRICS
    # ========================================================

    metrics = create_metrics()

    # ========================================================
    # ANSWER RELEVANCY
    # ========================================================

    await metrics["answer_relevancy"].a_measure(test_case)

    # ========================================================
    # FAITHFULNESS
    # ========================================================

    await metrics["faithfulness"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL RELEVANCY
    # ========================================================

    await metrics["contextual_relevancy"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL PRECISION
    # ========================================================

    await metrics["contextual_precision"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL RECALL
    # ========================================================

    await metrics["contextual_recall"].a_measure(test_case)

    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {
        "question": question,
        "expected_answer": expected_answer,
        "response": response,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,

        "answer_relevancy": metrics["answer_relevancy"].score,
        "answer_relevancy_reason": metrics["answer_relevancy"].reason,

        "faithfulness": metrics["faithfulness"].score,
        "faithfulness_reason": metrics["faithfulness"].reason,

        "contextual_relevancy": metrics["contextual_relevancy"].score,
        "contextual_relevancy_reason": metrics["contextual_relevancy"].reason,

        "contextual_precision": metrics["contextual_precision"].score,
        "contextual_precision_reason": metrics["contextual_precision"].reason,

        "contextual_recall": metrics["contextual_recall"].score,
        "contextual_recall_reason": metrics["contextual_recall"].reason,
    }

---
## 11. Batch Evaluation Runner with Progress Tracking

The function `run_deepeval_evaluation(dataset)` iterates through all test cases in the golden dataset with a visual `tqdm` progress bar, accumulating scores and diagnostic rationales.

In [20]:
async def run_deepeval_evaluation(dataset):

    results = []

    for row in tqdm(
        dataset,
        desc="Running DeepEval evaluation",
    ):

        result = await evaluate_test_case(row)
        results.append(result)

    return results

---
## 12. Main Orchestration, Score Aggregation & Diagnostic Reporting

The `main()` function handles the complete evaluation lifecycle:
1. **Runs batch evaluation**: `await run_deepeval_evaluation(dataset)`.
2. **Tabulates results**: Converts results into a pandas `DataFrame`.
3. **Calculates aggregate means**: Computes the average score across all 5 RAG dimensions.
4. **Exports results to CSV**: Saves the full report to `evals/deepeval/deepeval_langchain_results.csv` for CI/CD tracking.
5. **Prints qualitative reasons**: Displays the judge's detailed reasoning for every metric and question to pinpoint retrieval noise or generation flaws.

In [21]:
# ============================================================
# 16. MAIN FUNCTION
# ============================================================

async def main():

    print("\n" + "=" * 100)
    print("DEEPEVAL + LANGCHAIN RAG EVALUATION")
    print("=" * 100)

    # ========================================================
    # LOAD DATASET
    # ========================================================

    dataset = load_dataset()

    print("\nDataset loaded.")

    # ========================================================
    # RUN DEEPEVAL EVALUATION
    # ========================================================

    results = await run_deepeval_evaluation(dataset)

    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "answer_relevancy",
        "faithfulness",
        "contextual_relevancy",
        "contextual_precision",
        "contextual_recall",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CALCULATE AVERAGE SCORES
    # ========================================================

    average_answer_relevancy = df["answer_relevancy"].mean()
    average_faithfulness = df["faithfulness"].mean()
    average_contextual_relevancy = df["contextual_relevancy"].mean()
    average_contextual_precision = df["contextual_precision"].mean()
    average_contextual_recall = df["contextual_recall"].mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Answer Relevancy      : {average_answer_relevancy:.2f}")
    print(f"Average Faithfulness          : {average_faithfulness:.2f}")
    print(f"Average Contextual Relevancy  : {average_contextual_relevancy:.2f}")
    print(f"Average Contextual Precision  : {average_contextual_precision:.2f}")
    print(f"Average Contextual Recall     : {average_contextual_recall:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    experiment_directory = Path("evals/deepeval")
    experiment_directory.mkdir(parents=True, exist_ok=True)

    output_file = experiment_directory / "deepeval_langchain_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nRESPONSE:")
        print(result_row["response"])

        print("\nRETRIEVED DOCUMENT IDS:")
        print(result_row["retrieved_document_ids"])

        print("\nANSWER RELEVANCY:")
        print(result_row["answer_relevancy"])

        print("\nANSWER RELEVANCY REASON:")
        print(result_row["answer_relevancy_reason"])

        print("\nFAITHFULNESS:")
        print(result_row["faithfulness"])

        print("\nFAITHFULNESS REASON:")
        print(result_row["faithfulness_reason"])

        print("\nCONTEXTUAL RELEVANCY:")
        print(result_row["contextual_relevancy"])

        print("\nCONTEXTUAL RELEVANCY REASON:")
        print(result_row["contextual_relevancy_reason"])

        print("\nCONTEXTUAL PRECISION:")
        print(result_row["contextual_precision"])

        print("\nCONTEXTUAL PRECISION REASON:")
        print(result_row["contextual_precision_reason"])

        print("\nCONTEXTUAL RECALL:")
        print(result_row["contextual_recall"])

        print("\nCONTEXTUAL RECALL REASON:")
        print(result_row["contextual_recall_reason"])

        print("\n" + "-" * 100)

### 12.1 Execution Trigger
Executes the asynchronous `main()` function.

In [27]:
await main()


DEEPEVAL + LANGCHAIN RAG EVALUATION

Dataset loaded.

DETAILED RESULTS
                      question  answer_relevancy  faithfulness  contextual_relevancy  contextual_precision  contextual_recall
                What is Ragas?               1.0           1.0                   1.0                   1.0                1.0
How do Ragas experiments work?               1.0           1.0                   1.0                   1.0                1.0
     What do Ragas metrics do?               1.0           1.0                   1.0                   1.0                1.0
                  What is RAG?               1.0           1.0                   1.0                   1.0                1.0

SUMMARY
Average Answer Relevancy      : 1.00
Average Faithfulness          : 1.00
Average Contextual Relevancy  : 1.00
Average Contextual Precision  : 1.00
Average Contextual Recall     : 1.00

Results saved to:
evals/deepeval/deepeval_langchain_results.csv

DETAILED EVALUATION REASONS

QUESTION: